# Multinomial classification notebook


In [13]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, log_loss
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

PANEL_START_YEAR = 1970
PANEL_PATH = Path("modeling_panels") / f"model_panel_start_{PANEL_START_YEAR}_01_31.parquet"
TEST_FRACTION = 0.25
CV_SPLITS = 4
MAX_ITER = 5000
RANDOM_STATE = 42
C_GRID = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

panel_monthly = pd.read_parquet(PANEL_PATH).copy()
if "date" not in panel_monthly.columns:
    panel_monthly.index = pd.to_datetime(panel_monthly.index)
    panel_monthly = panel_monthly.reset_index().rename(columns={panel_monthly.columns[0]: "date"})
else:
    panel_monthly["date"] = pd.to_datetime(panel_monthly["date"])
panel_monthly = panel_monthly.sort_values("date").reset_index(drop=True)

target_cols = sorted([c for c in panel_monthly.columns if "label_" in c])
EXCLUDE_PATTERNS = ["label_", "_fwd_ret_", "_fwd_excess_", "_exp_q"]
feature_cols = [c for c in panel_monthly.columns if c != "date" and not any(p in c for p in EXCLUDE_PATTERNS)]

print("Panel:", PANEL_PATH)
print("Shape:", panel_monthly.shape)
print("Date range:", panel_monthly["date"].min(), "->", panel_monthly["date"].max())
print("Targets:", len(target_cols))
print("Features:", len(feature_cols))

def compute_metrics(y_true, pred, proba, labels):
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "log_loss": float(log_loss(y_true, proba, labels=labels)),
    }

def build_target_dataset(target_col):
    tmp = panel_monthly[["date", target_col] + feature_cols].copy()
    tmp = tmp.loc[tmp[target_col].notna()].sort_values("date").reset_index(drop=True)

    classes = sorted(tmp[target_col].dropna().unique().tolist())
    if len(classes) < 2:
        return None, None, None, None, None

    label_map = {lab: i for i, lab in enumerate(classes)}
    tmp["y"] = tmp[target_col].map(label_map).astype(int)

    n_obs = len(tmp)
    test_size = int(np.ceil(TEST_FRACTION * n_obs))
    train_size = n_obs - test_size

    train_df = tmp.iloc[:train_size].copy()
    test_df = tmp.iloc[train_size:].copy()

    if train_df["y"].nunique() < 2 or test_df["y"].nunique() < 1:
        return None, None, None, None, None

    return tmp, train_df, test_df, classes, train_size

def fit_l2_for_target(X_train, y_train, n_classes, c_grid, max_iter, random_state, cv_splits):
    multi_class_mode = "auto" if n_classes == 2 else "multinomial"
    global_labels = list(range(n_classes))

    def make_pipe(C):
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                C=C,
                multi_class=multi_class_mode,
                solver="lbfgs",
                max_iter=max_iter,
                random_state=random_state,
            )),
        ])

    rows = []
    cv = TimeSeriesSplit(n_splits=cv_splits)

    for C in c_grid:
        losses, accs, baccs = [], [], []

        for tr_idx, va_idx in cv.split(X_train):
            X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
            y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

            if pd.Series(y_tr).nunique() < 2:
                continue

            pipe = make_pipe(C)
            pipe.fit(X_tr, y_tr)

            proba_fold = pipe.predict_proba(X_va)
            pred_va = pipe.predict(X_va)

            model_classes = list(pipe.named_steps["clf"].classes_)

            proba_full = np.zeros((len(X_va), n_classes), dtype=float)
            for j, cls in enumerate(model_classes):
                proba_full[:, cls] = proba_fold[:, j]

            losses.append(log_loss(y_va, proba_full, labels=global_labels))
            accs.append(accuracy_score(y_va, pred_va))
            baccs.append(balanced_accuracy_score(y_va, pred_va))

        if len(losses) == 0:
            continue

        rows.append({
            "C": C,
            "cv_log_loss": np.mean(losses),
            "cv_acc": np.mean(accs),
            "cv_bacc": np.mean(baccs),
        })

    if len(rows) == 0:
        raise ValueError("No valid CV folds for this target.")

    cv_df = pd.DataFrame(rows).sort_values(
        ["cv_log_loss", "cv_acc"],
        ascending=[True, False]
    ).reset_index(drop=True)

    best_C = float(cv_df.iloc[0]["C"])
    model = make_pipe(best_C)
    model.fit(X_train, y_train)

    return model, best_C

def get_group(target_col):
    if "joint_label_q2xq2" in target_col:
        return "joint_q2xq2"
    if "joint_label_q3xq3" in target_col:
        return "joint_q3xq3"
    if "_label_q2_" in target_col:
        return "q2"
    if "_label_q3_" in target_col:
        return "q3"
    if "_label_q4_" in target_col:
        return "q4"
    return "other"

def get_asset(target_col):
    if target_col.startswith("sp500_"):
        return "sp500"
    if target_col.startswith("bond_"):
        return "bond"
    return "other"

def get_proxy_candidates(target_col):
    asset = get_asset(target_col)
    if asset == "sp500":
        cand = ["sprtrn_sp500", "vwretd", "mktrf", "smb", "hml", "rv_20d", "rv_60d", "VIXCLSx", "bond_10y_drawdown"]
    elif asset == "bond":
        cand = ["agg_ret", "bond_10y_ret", "bond_10y_mom_12m", "bond_10y_drawdown", "rv_20d", "rv_60d", "VIXCLSx", "credit_spread_baa_aaa", "derived_baa_aaa"]
    else:
        cand = []
    return [c for c in cand if c in panel_monthly.columns]

Panel: modeling_panels\model_panel_start_1970_01_31.parquet
Shape: (675, 274)
Date range: 1970-01-31 00:00:00 -> 2026-03-31 00:00:00
Targets: 48
Features: 219


In [14]:
all_rows = []

for target_col in target_cols:
    built = build_target_dataset(target_col)
    if built[0] is None:
        continue

    tmp, train_df, test_df, classes, train_size = built

    X_tr = train_df[feature_cols]
    y_tr = train_df["y"].astype(int)
    X_te = test_df[feature_cols]
    y_te = test_df["y"].astype(int)

    try:
        model, best_C = fit_l2_for_target(
            X_train=X_tr,
            y_train=y_tr,
            n_classes=len(classes),
            c_grid=C_GRID,
            max_iter=MAX_ITER,
            random_state=RANDOM_STATE,
            cv_splits=CV_SPLITS,
        )

        proba_fold = model.predict_proba(X_te)
        pred = model.predict(X_te)

        model_classes = list(model.named_steps["clf"].classes_)
        proba_full = np.zeros((len(X_te), len(classes)), dtype=float)
        for j, cls in enumerate(model_classes):
            proba_full[:, cls] = proba_fold[:, j]

        all_rows.append({
            "target_col": target_col,
            "target_group": get_group(target_col),
            "n_obs": len(tmp),
            "n_classes": len(classes),
            "best_C": best_C,
            "test_accuracy": accuracy_score(y_te, pred),
            "test_balanced_accuracy": balanced_accuracy_score(y_te, pred),
            "test_log_loss": log_loss(y_te, proba_full, labels=list(range(len(classes)))),
        })

    except Exception as e:
        print(f"Skipping {target_col}: {e}")

all_target_results_df = pd.DataFrame(all_rows).sort_values(
    ["test_log_loss", "test_accuracy"],
    ascending=[True, False]
).reset_index(drop=True)

family_summary_df = (
    all_target_results_df
    .groupby("target_group", as_index=False)
    .agg(
        n_targets=("target_col", "count"),
        mean_test_accuracy=("test_accuracy", "mean"),
        mean_test_bal_acc=("test_balanced_accuracy", "mean"),
        mean_test_log_loss=("test_log_loss", "mean"),
    )
    .sort_values("mean_test_log_loss")
    .reset_index(drop=True)
)

best_targets_df = all_target_results_df[
    ["target_col", "n_classes", "test_accuracy", "test_balanced_accuracy", "test_log_loss"]
].head(3)

worst_targets_df = (
    all_target_results_df[
        ["target_col", "n_classes", "test_accuracy", "test_balanced_accuracy", "test_log_loss"]
    ]
    .tail(2)
    .sort_values(["test_log_loss", "test_accuracy"], ascending=[False, True])
    .reset_index(drop=True)
)

display(family_summary_df)
display(best_targets_df)
display(worst_targets_df)

,target_group,n_targets,mean_test_accuracy,mean_test_bal_acc,mean_test_log_loss
0,q2,12,0.498519,0.510645,0.696528
1,q3,12,0.382561,0.338793,1.107756
2,joint_q2xq2,6,0.250217,0.260886,1.411364
3,q4,12,0.282789,0.248299,1.411465
4,joint_q3xq3,6,0.158900,0.107011,2.204860


,target_col,n_classes,test_accuracy,test_balanced_accuracy,test_log_loss
0,bond_label_q2_expanding_1m,2,0.580882,0.51893,0.677494
1,sp500_label_q2_expanding_6m,2,0.652439,0.49537,0.679767
2,sp500_label_q2_expanding_1m,2,0.581818,0.50000,0.681695


,target_col,n_classes,test_accuracy,test_balanced_accuracy,test_log_loss
0,joint_label_q3xq3_full_6m,9,0.034247,0.111111,2.275396
1,joint_label_q3xq3_full_3m,9,0.040816,0.047619,2.244380


In [15]:
MAIN_BINARY_Q2_TARGETS = [
    "bond_label_q2_expanding_1m",
    "bond_label_q2_full_1m",
    "sp500_label_q2_expanding_1m",
    "sp500_label_q2_full_1m",
    "bond_label_q2_expanding_6m",
    "bond_label_q2_full_6m",
    "sp500_label_q2_expanding_6m",
    "sp500_label_q2_full_6m",
]
MAIN_BINARY_Q2_TARGETS = [t for t in MAIN_BINARY_Q2_TARGETS if t in panel_monthly.columns]

baseline_rows = []

for target_col in MAIN_BINARY_Q2_TARGETS:
    built = build_target_dataset(target_col)
    if built[0] is None:
        continue

    tmp, train_df, test_df, classes, train_size = built

    X_tr = train_df[feature_cols]
    y_tr = train_df["y"].astype(int)
    X_te = test_df[feature_cols]
    y_te = test_df["y"].astype(int)
    labels = list(range(len(classes)))

    model, best_C = fit_l2_for_target(
        X_train=X_tr,
        y_train=y_tr,
        n_classes=len(classes),
        c_grid=C_GRID,
        max_iter=MAX_ITER,
        random_state=RANDOM_STATE,
        cv_splits=CV_SPLITS,
    )

    l2_proba_fold = model.predict_proba(X_te)
    l2_pred = model.predict(X_te)

    model_classes = list(model.named_steps["clf"].classes_)
    l2_proba = np.zeros((len(X_te), len(classes)), dtype=float)
    for j, cls in enumerate(model_classes):
        l2_proba[:, cls] = l2_proba_fold[:, j]

    baseline_rows.append({
        "target_col": target_col,
        "model": "logistic_l2",
        **compute_metrics(y_te, l2_pred, l2_proba, labels)
    })

    maj = int(y_tr.value_counts().idxmax())
    maj_pred = np.full(len(y_te), maj, dtype=int)
    maj_proba = np.zeros((len(y_te), len(classes)))
    maj_proba[:, maj] = 1.0
    baseline_rows.append({
        "target_col": target_col,
        "model": "majority_class",
        **compute_metrics(y_te, maj_pred, maj_proba, labels)
    })

    freq = y_tr.value_counts(normalize=True).sort_index()
    freq_vec = np.array([freq.get(i, 0.0) for i in range(len(classes))], dtype=float)
    freq_pred = np.full(len(y_te), int(np.argmax(freq_vec)), dtype=int)
    freq_proba = np.tile(freq_vec, (len(y_te), 1))
    baseline_rows.append({
        "target_col": target_col,
        "model": "class_frequency",
        **compute_metrics(y_te, freq_pred, freq_proba, labels)
    })

    tmp2 = tmp.copy()
    tmp2["y_lag1"] = tmp2["y"].shift(1)
    usable = tmp2.iloc[train_size:].loc[tmp2.iloc[train_size:]["y_lag1"].notna()].copy()

    pers_y = usable["y"].astype(int).to_numpy()
    pers_pred = usable["y_lag1"].astype(int).to_numpy()
    pers_proba = np.zeros((len(pers_pred), len(classes)))
    if len(pers_pred):
        pers_proba[np.arange(len(pers_pred)), pers_pred] = 1.0

    baseline_rows.append({
        "target_col": target_col,
        "model": "persistence_lag1",
        **compute_metrics(pers_y, pers_pred, pers_proba, labels)
    })

q2_baseline_df = pd.DataFrame(baseline_rows)

q2_compare_df = (
    q2_baseline_df.loc[q2_baseline_df["model"] == "logistic_l2", ["target_col", "accuracy"]]
    .rename(columns={"accuracy": "l2_acc"})
    .merge(
        q2_baseline_df.loc[q2_baseline_df["model"] == "persistence_lag1", ["target_col", "accuracy"]]
        .rename(columns={"accuracy": "persistence_acc"}),
        on="target_col"
    )
    .merge(
        q2_baseline_df.loc[q2_baseline_df["model"] == "class_frequency", ["target_col", "accuracy"]]
        .rename(columns={"accuracy": "freq_acc"}),
        on="target_col"
    )
    .merge(
        q2_baseline_df.loc[q2_baseline_df["model"] == "majority_class", ["target_col", "accuracy"]]
        .rename(columns={"accuracy": "majority_acc"}),
        on="target_col"
    )
)

l2_beats_persistence = int((q2_compare_df["l2_acc"] > q2_compare_df["persistence_acc"]).sum())
l2_beats_freq = int((q2_compare_df["l2_acc"] > q2_compare_df["freq_acc"]).sum())
l2_beats_majority = int((q2_compare_df["l2_acc"] > q2_compare_df["majority_acc"]).sum())

display(q2_compare_df)
print("L2 beats persistence on", l2_beats_persistence, "/", len(q2_compare_df))
print("L2 beats class-frequency on", l2_beats_freq, "/", len(q2_compare_df))
print("L2 beats majority-class on", l2_beats_majority, "/", len(q2_compare_df))

,target_col,l2_acc,persistence_acc,freq_acc,majority_acc
0,bond_label_q2_expanding_1m,0.580882,0.470588,0.433824,0.433824
1,bond_label_q2_full_1m,0.626667,0.466667,0.446667,0.446667
2,sp500_label_q2_expanding_1m,0.581818,0.466667,0.418182,0.418182
3,sp500_label_q2_full_1m,0.569697,0.466667,0.430303,0.430303
4,bond_label_q2_expanding_6m,0.365672,0.835821,0.343284,0.343284
5,bond_label_q2_full_6m,0.402685,0.825503,0.402685,0.402685
6,sp500_label_q2_expanding_6m,0.652439,0.817073,0.341463,0.341463
7,sp500_label_q2_full_6m,0.402439,0.804878,0.353659,0.353659


L2 beats persistence on 4 / 8
L2 beats class-frequency on 7 / 8
L2 beats majority-class on 7 / 8


In [16]:
COMPARE_TARGETS = [
    "bond_label_q2_expanding_1m", "bond_label_q2_full_1m", "sp500_label_q2_expanding_1m", "sp500_label_q2_full_1m",
    "bond_label_q2_expanding_6m", "bond_label_q2_full_6m", "sp500_label_q2_expanding_6m", "sp500_label_q2_full_6m",
    "bond_label_q3_expanding_1m", "bond_label_q3_full_1m", "sp500_label_q3_expanding_1m", "sp500_label_q3_full_1m",
    "bond_label_q3_expanding_6m", "bond_label_q3_full_6m", "sp500_label_q3_expanding_6m", "sp500_label_q3_full_6m",
    "bond_label_q4_expanding_1m", "bond_label_q4_full_1m", "sp500_label_q4_expanding_1m", "sp500_label_q4_full_1m",
    "bond_label_q4_expanding_6m", "bond_label_q4_full_6m", "sp500_label_q4_expanding_6m", "sp500_label_q4_full_6m",
]
COMPARE_TARGETS = [t for t in COMPARE_TARGETS if t in panel_monthly.columns]
compare_rows = []
for target_col in COMPARE_TARGETS:
    tmp, train_df, test_df, classes, train_size = build_target_dataset(target_col)
    tmp["y_lag1_feature"] = tmp["y"].shift(1)
    proxy_lag_cols = []
    for col in get_proxy_candidates(target_col):
        lag_col = f"{col}_lag1_feature"
        tmp[lag_col] = tmp[col].shift(1)
        proxy_lag_cols.append(lag_col)
    train_aug = tmp.iloc[:train_size].copy()
    test_aug = tmp.iloc[train_size:].copy()
    for model_name, feat_set in [
        ("logistic_l2_base", feature_cols.copy()),
        ("logistic_l2_plus_y_lag1", feature_cols + ["y_lag1_feature"]),
        ("logistic_l2_plus_state_proxies", feature_cols + proxy_lag_cols),
    ]:
        feat_set = list(dict.fromkeys(feat_set))
        X_tr = train_aug[feat_set]
        y_tr = train_aug["y"].astype(int)
        X_te = test_aug[feat_set]
        y_te = test_aug["y"].astype(int)
        model, best_C = fit_l2_for_target(
    X_train=X_tr,
    y_train=y_tr,
    n_classes=len(classes),
    c_grid=C_GRID,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
    cv_splits=CV_SPLITS,
)
        proba = model.predict_proba(X_te)
        pred = proba.argmax(axis=1)
        compare_rows.append({"target_col": target_col, "target_group": get_group(target_col), "model": model_name, "accuracy": float(accuracy_score(y_te, pred))})

compare_df = pd.DataFrame(compare_rows)
compare_wide = compare_df.pivot_table(index=["target_col", "target_group"], columns="model", values="accuracy").reset_index()
compare_wide["ylag_minus_base"] = compare_wide["logistic_l2_plus_y_lag1"] - compare_wide["logistic_l2_base"]
compare_wide["proxy_minus_base"] = compare_wide["logistic_l2_plus_state_proxies"] - compare_wide["logistic_l2_base"]
lag_group_summary_df = compare_wide.groupby("target_group", as_index=False).agg(base=("logistic_l2_base", "mean"), ylag=("logistic_l2_plus_y_lag1", "mean"), proxy=("logistic_l2_plus_state_proxies", "mean"))
lag_group_summary_df["ylag_minus_base"] = lag_group_summary_df["ylag"] - lag_group_summary_df["base"]
lag_group_summary_df["proxy_minus_base"] = lag_group_summary_df["proxy"] - lag_group_summary_df["base"]
ylag_improved = compare_wide.loc[compare_wide["ylag_minus_base"] > 0, ["target_col", "logistic_l2_base", "logistic_l2_plus_y_lag1"]]
proxy_improved = compare_wide.loc[compare_wide["proxy_minus_base"] > 0, ["target_col", "logistic_l2_base", "logistic_l2_plus_state_proxies"]]
display(lag_group_summary_df)
display(ylag_improved)
display(proxy_improved)
print("REPORT")
print("- q2 is the only target family that is even moderately usable under plain L2 logistic.")
print("- q3 is weak, q4 is weaker, and joint q3xq3 is effectively unusable.")
print(f"- L2 beats persistence on {l2_beats_persistence}/8 q2 targets.")
print(f"- L2 beats class-frequency on {l2_beats_freq}/8 q2 targets.")
print(f"- L2 beats majority-class on {l2_beats_majority}/8 q2 targets.")
print("- Adding y_{t-1} helps selectively, mainly on some longer-horizon targets.")
print("- Adding lagged observable state proxies helps less consistently than y_{t-1}.")


,target_group,base,ylag,proxy,ylag_minus_base,proxy_minus_base
0,q2,0.522787,0.532101,0.517294,0.009314,-0.005493
1,q3,0.370988,0.375945,0.373265,0.004957,0.002277
2,q4,0.294284,0.304463,0.297401,0.010178,0.003116


model,target_col,logistic_l2_base,logistic_l2_plus_y_lag1
1,bond_label_q2_expanding_6m,0.365672,0.455224
7,bond_label_q3_full_6m,0.187919,0.221477
9,bond_label_q4_expanding_6m,0.238806,0.253731
11,bond_label_q4_full_6m,0.308725,0.369128
13,sp500_label_q2_expanding_6m,0.652439,0.658537
15,sp500_label_q2_full_6m,0.402439,0.414634
17,sp500_label_q3_expanding_6m,0.439024,0.445122
23,sp500_label_q4_full_6m,0.231707,0.237805


model,target_col,logistic_l2_base,logistic_l2_plus_state_proxies
1,bond_label_q2_expanding_6m,0.365672,0.388060
11,bond_label_q4_full_6m,0.308725,0.315436
16,sp500_label_q3_expanding_1m,0.357576,0.363636
18,sp500_label_q3_full_1m,0.363636,0.369697
19,sp500_label_q3_full_6m,0.432927,0.439024
20,sp500_label_q4_expanding_1m,0.315152,0.321212
22,sp500_label_q4_full_1m,0.309091,0.315152
23,sp500_label_q4_full_6m,0.231707,0.237805


REPORT
- q2 is the only target family that is even moderately usable under plain L2 logistic.
- q3 is weak, q4 is weaker, and joint q3xq3 is effectively unusable.
- L2 beats persistence on 4/8 q2 targets.
- L2 beats class-frequency on 7/8 q2 targets.
- L2 beats majority-class on 7/8 q2 targets.
- Adding y_{t-1} helps selectively, mainly on some longer-horizon targets.
- Adding lagged observable state proxies helps less consistently than y_{t-1}.
